In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score
from utils import utils_ml 

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.feature_selection import VarianceThreshold

In [2]:
feature_names = np.load("feature_names.npy",allow_pickle=True)
X_arr = np.load("features_filtered.npy", mmap_mode='r')


In [3]:
# Load and concatenate all features
file_prefix = "/home/felix/vscodeProjects/pole-learning/data_35_sim_1/rawFeatures/P"
file_suffix = "_intensity.pkl"
num_files = 35

class_to_poles = {
    0: [0, 0, 0],  # 1 pole on [bt]
    1: [1, 0, 0],  # 1 pole on [bb]
    2: [0, 1, 0],  # 1 pole on [tb]
    3: [0, 0, 1],  # 1 pole on [bt] and 1 pole on [bb]
    4: [2, 0, 0],  # 1 pole on [bb] and 1 pole on [tb]
    5: [0, 2, 0],  # 1 pole on each of [bt], [bb], and [tb]
    6: [0, 0, 2],  # 2 poles on [bb] and 1 pole on [tb]
    7: [1, 1, 0],   # 1 pole on [bb] and 2 poles on [tb]
    8: [1, 0, 1],
    9: [0, 1, 1],
    10: [3, 0, 0],
    11: [0, 3, 0],
    12: [0, 0, 3],
    13: [2, 1, 0],
    14: [2, 0, 1],
    15: [1, 2, 0],
    16: [0, 2, 1],
    17: [1, 0, 2],
    18: [0, 1, 2],
    19: [1, 1, 1],
    20: [4, 0, 0],
    21: [0, 4, 0],
    22: [0, 0, 4],
    23: [3, 1, 0],
    24: [3, 0, 1],
    25: [1, 3, 0],
    26: [0, 3, 1],
    27: [1, 0, 3],
    28: [0, 1, 3],
    29: [2, 2, 0],
    30: [2, 0, 2],
    31: [0, 2, 2],
    32: [2, 1, 1],
    33: [1, 2, 1],
    34: [1, 1, 2],
}


y_arr_classification = np.array([np.tile(i,np.load(f"{file_prefix}{0:02d}{file_suffix}", allow_pickle=True).shape[0]) for i in np.arange(num_files)]).flatten()
y_arr_regression = utils_ml.convert_labels(y_arr_classification, class_to_poles)*1.0

In [4]:
rg = CatBoostRegressor(
    iterations=500,
    verbose=100,
    loss_function="MultiRMSE",
    early_stopping_rounds=10 
    )
X_train, X_val, y_train, y_val = train_test_split(X_arr, y_arr_regression, random_state=42, test_size=0.2)
rg.fit(X_train, y_train, eval_set=(X_val,y_val), plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 1.8658708	test: 1.8660420	best: 1.8660420 (0)	total: 355ms	remaining: 2m 57s
100:	learn: 1.0652817	test: 1.0631812	best: 1.0631812 (100)	total: 30.7s	remaining: 2m 1s
200:	learn: 0.9942429	test: 0.9934744	best: 0.9934744 (200)	total: 58.7s	remaining: 1m 27s
300:	learn: 0.9625099	test: 0.9631647	best: 0.9631647 (300)	total: 1m 27s	remaining: 57.8s
400:	learn: 0.9429017	test: 0.9447486	best: 0.9447486 (400)	total: 1m 55s	remaining: 28.6s
499:	learn: 0.9276697	test: 0.9308028	best: 0.9308028 (499)	total: 2m 24s	remaining: 0us

bestTest = 0.9308028418
bestIteration = 499



In [5]:
accuracy_score(utils_ml.reconvert_labels(np.abs(np.round(rg.predict(X_val))), class_to_poles), utils_ml.reconvert_labels(y_val, class_to_poles))

0.5164428571428571

In [6]:
importances = rg.get_feature_importance()
# feature_names = X.columns
sorted_idx = np.argsort(importances)[::-1]
X_arr_sorted= X_arr[:,sorted_idx]
names = feature_names[sorted_idx]

In [7]:



feature_importances = {}

for n,v in zip(names[:128],np.sort(importances)[::-1][:128]):
    feature_importances[n] = v


In [8]:
import pickle

with open('feature_importances_T11T12NoBg.pkl', 'wb') as f:
    pickle.dump(feature_importances, f)


In [9]:
stop

NameError: name 'stop' is not defined

array(['intensity__approximate_entropy__m_2__r_0.5',
       'intensity__agg_autocorrelation__f_agg_"var"__maxlag_40',
       'intensity__last_location_of_maximum',
       'intensity__approximate_entropy__m_2__r_0.3',
       'intensity__autocorrelation__lag_9',
       'intensity__agg_linear_trend__attr_"slope"__chunk_len_50__f_agg_"max"',
       'intensity__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.0',
       'intensity__fft_coefficient__attr_"real"__coeff_1',
       'intensity__approximate_entropy__m_2__r_0.7',
       'intensity__cwt_coefficients__coeff_1__w_10__widths_(2, 5, 10, 20)',
       'intensity__fft_coefficient__attr_"imag"__coeff_2',
       'intensity__autocorrelation__lag_8',
       'intensity__energy_ratio_by_chunks__num_segments_10__segment_focus_6',
       'intensity__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.2',
       'intensity__first_location_of_maximum',
       'intensity__ratio_beyond_r_sigma__r_2', 'intensity__skewness',
       'inte

In [ ]:
features_selected = X_arr_sorted[0,:128]

features = np.load("features_filtered.npy", mmap_mode='r')[0]

# Convert to list for easier index matching
features_list = features.tolist()
selected_names = []
cnt = 0

for val in features_selected:
    # Find all indices where the value matches (in case of duplicates)
    matches = [i for i, x in enumerate(features_list) if np.round(x,10) == np.round(val,10)]

    # If unique match, grab the name
    if len(matches) == 1:
        selected_names.append(feature_names[matches[0]])
    else:
        cnt += 1

print("Selected feature names:", selected_names)


Selected feature names: ['intensity__approximate_entropy__m_2__r_0.5', 'intensity__agg_autocorrelation__f_agg_"var"__maxlag_40', 'intensity__approximate_entropy__m_2__r_0.3', 'intensity__autocorrelation__lag_9', 'intensity__agg_linear_trend__attr_"slope"__chunk_len_50__f_agg_"max"', 'intensity__fft_coefficient__attr_"real"__coeff_1', 'intensity__approximate_entropy__m_2__r_0.7', 'intensity__cwt_coefficients__coeff_1__w_10__widths_(2, 5, 10, 20)', 'intensity__fft_coefficient__attr_"imag"__coeff_2', 'intensity__autocorrelation__lag_8', 'intensity__energy_ratio_by_chunks__num_segments_10__segment_focus_6', 'intensity__first_location_of_maximum', 'intensity__skewness', 'intensity__sample_entropy', 'intensity__agg_linear_trend__attr_"slope"__chunk_len_50__f_agg_"var"', 'intensity__cwt_coefficients__coeff_14__w_20__widths_(2, 5, 10, 20)', 'intensity__energy_ratio_by_chunks__num_segments_10__segment_focus_5', 'intensity__cid_ce__normalize_True', 'intensity__autocorrelation__lag_7', 'intensity

In [ ]:
matches,cnt

([135, 150], 25)

In [ ]:
# np.save('features_selected.npy', X_arr[:,sorted_idx[:100]])

In [ ]:
# importances[sorted_idx][:100]

In [ ]:
# y_arr_regression[sorted_idx[:d]].shape

In [ ]:
# d_arr = [10,100,200,300,400,429]
# acc_arr = []
# for d in d_arr:
#     _rg = CatBoostRegressor(
#         verbose=100,
#         loss_function="MultiRMSE",
#         early_stopping_rounds=10 
#         )
#     X_train, X_val, y_train, y_val = train_test_split(X_arr[:,sorted_idx[:d]], y_arr_regression, random_state=42, test_size=0.2)
#     _rg.fit(X_train, y_train, eval_set=(X_val,y_val), plot=True)

#     acc_arr.append(accuracy_score(utils_ml.reconvert_labels(np.abs(np.round(_rg.predict(X_val))), class_to_poles), utils_ml.reconvert_labels(y_val, class_to_poles)))
stop

NameError: name 'stop' is not defined

In [ ]:
import numpy as np
import gc
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


feature_percentages = np.logspace(-2, 0, 5)
feature_counts = np.unique(np.clip((feature_percentages * X_arr_raw.shape[1]).astype(int), 1, X_arr_raw.shape[1]))

# Dictionary to hold fold accuracies per feature count
acc_dict = {}
fold_accuracies = []
for d in feature_counts:
    print(f"\nEvaluating {d} features ({d/75:.1%} of total)...")
    

    # kf = KFold(n_splits=5, shuffle=True, random_state=42)
    # for fold, (train_index, val_index) in enumerate(kf.split(X_arr)):
    X_train, X_val = X_arr_raw[:, :d], X_arr_raw[:, :d]
    y_train, y_val = y_arr_regression, y_arr_regression

    model = CatBoostRegressor(
        iterations=1000,
        verbose=0,
        loss_function="MultiRMSE",
        early_stopping_rounds=10
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

    # Save model
    # model_path = f"savedModels/catboost_model_{d}_fold{fold}.cbm"
    # model.save_model(model_path)

    # Predict and evaluate
    y_pred = np.abs(np.round(model.predict(X_val)))
    acc = accuracy_score(
        utils_ml.reconvert_labels(y_pred, class_to_poles),
        utils_ml.reconvert_labels(y_val, class_to_poles)
    )
    fold_accuracies.append(acc)

    # Clean up
    del model
    gc.collect()

acc_dict[d] = fold_accuracies
print(f"Fold Accuracies: {fold_accuracies}")

# Plotting mean ± std as error bars
# means = [np.mean(acc_dict[d]) for d in feature_counts]
# stds = [np.std(acc_dict[d]) for d in feature_counts]

# plt.figure(figsize=(4, 3))
# plt.errorbar(100 * feature_counts / 429, means, yerr=stds, fmt='-o', capsize=5)
# plt.xscale('log')
# plt.xticks([1, 3, 10, 30, 100], labels=["1%", "3%", "10%", "30%", "100%"])
# plt.xlabel("Percentage of Features (log scale)")
# plt.ylabel("Validation Accuracy")
# plt.title("Effect of Feature Count on Accuracy")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


Evaluating 1 features (1.3% of total)...

Evaluating 2 features (2.7% of total)...

Evaluating 7 features (9.3% of total)...

Evaluating 23 features (30.7% of total)...

Evaluating 75 features (100.0% of total)...
Fold Accuracies: [0.12864285714285714, 0.13929142857142857, 0.3277685714285714, 0.43968, 0.45112]


In [ ]:
fold_accuracies

[0.45310285714285714]

In [ ]:
clf = CatBoostClassifier(
    iterations=100
    loss_function="MultiClass",
    class_weights=None,
    auto_class_weights=None,
    verbose=100,
    early_stopping_rounds=10,        
)
X_train, X_val, y_train, y_val = train_test_split(X_arr, y_arr_classification,random_state=42, test_size=0.2)
clf.fit(X_arr[:,:100], y_arr_classification, eval_set=(X_val[:,:100],y_val),plot=True)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1381764609.py, line 2)

In [ ]:
accuracy_score(clf.predict(X_val),y_val)

CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

In [ ]:
# percentages = [0.5,0.75,0.9]
# num_features = X_train.shape[1]

# feature_importances = clf.get_feature_importance()
# feature_names = [f'Feature {i}' for i in range(X_train.shape[1])]

# # Rank features by importance
# ranked_features = sorted(zip(feature_names, feature_importances), key=lambda x: x[1], reverse=True)

# # Store results
# results = {}

# # Train and evaluate for each percentage
# for percent in percentages:
#     k = max(1, int(percent * num_features))  # Number of features to select
#     top_features = [name for name, importance in ranked_features[:k]]
#     selected_columns = [feature_names.index(name) for name in top_features]
    
#     # Reduce dataset
#     X_train_selected = X_train[:, selected_columns]
#     X_val_selected = X_val[:, selected_columns]
    
#     # Train with selected features
#     clf_reduced = CatBoostClassifier(verbose=100)
#     clf_reduced.fit(X_train_selected, y_train)
    
#     # Evaluate accuracy
#     y_pred = clf_reduced.predict(X_val_selected)
#     acc = accuracy_score(y_val, y_pred)
#     results[percent] = acc

# print("Accuracy with top features:")
# for percent, acc in results.items():
#     print(f"Top {int(percent * 100)}% features: {acc:.4f}")

In [ ]:

# # Define a range of thresholds to test
# thresholds = np.arange(1, X_train.shape[1], 5)  # 50 thresholds between 0 and 0.5
# accuracies = []
# num_features_list = []

# best_accuracy = 0
# best_threshold = 0
# best_num_features = X_train.shape[1]

# for threshold in thresholds:
#     selector = SelectKBest(f_classif, k=int(threshold))
#     X_train_selected = selector.fit_transform(X_train, y_train)
#     X_test_selected = selector.transform(X_val)
    
#     # Initialize and fit a HistGradientBoostingClassifier with isotonic calibration
#     clf_reduced = CatBoostClassifier(verbose=100)
#     clf_reduced.fit(X_train_selected, y_train)
    
#     # Evaluate accuracy
#     y_pred = clf_reduced.predict(X_test_selected)
#     accuracy = accuracy_score(y_val, y_pred)
    
#     # Keep track of accuracy and number of features
#     num_features = X_train_selected.shape[1]
#     accuracies.append(accuracy)
#     num_features_list.append(num_features)
    
#     # Keep track of the best threshold based on accuracy and feature count
#     if accuracy > best_accuracy or (accuracy == best_accuracy and num_features < best_num_features):
#         best_accuracy = accuracy
#         best_threshold = threshold
#         best_num_features = num_features

# # Plot threshold vs number of features
# plt.figure(figsize=(10, 6))
# plt.plot(num_features_list, accuracies, label="Number of Features", color='blue', marker='o')
# plt.ylabel("Accuracy")
# plt.xlabel("Number of Features")
# plt.title("Threshold vs Number of Features")
# plt.grid(True)

# plt.show()

# # Print the best threshold and accuracy information
# print(f"Best Threshold: {best_threshold:.4f}")
# print(f"Best Accuracy: {best_accuracy:.4f}")
# print(f"Number of Features: {best_num_features}")
